# v8 GQA M-packing on TENSOR CORES — the gate (Cut 2a: Turing WMMA, Colab T4)

Cut 2a is the **GEMV→GEMM step on top of Cut 1's M-packing**: same paged GQA split-KV decode, but the QK and PV matmuls run on **16×16×16 WMMA tensor cores** (v5's fix). `M=G` is padded to 16 (rows ≥G zeroed). Tensor cores DON'T change bytes (`AI=2G/b` unchanged) — the bet is they **close part of Cut 1's per-CTA gap** (Cut 1 left %HBM ≤11%). Runs on the free T4; the A100 `mma.m16n8k16`+cp.async peak version is Cut 2b. **Headline check: does v8_gqa_tc beat v8_gqa (Cut 1)?**

## 0. Dependencies + GPU (venv-safe)

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    import torch  # already on the image? keep it, don't churn the version
except ImportError:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
pip('ninja', 'pytest', 'numpy')

# vast.ai venv fix: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!which python && python -c "import torch; print('shell python sees torch', torch.__version__)"

torch 2.11.0+cu128 | cuda 12.8 | cap (7, 5)
name, compute_cap
Tesla T4, 7.5
/usr/local/bin/python
shell python sees torch 2.11.0+cu128


## 1. Get the repo

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

cwd /content/flashattention-cuda


## 2. Roofline — UNCHANGED from Cut 1 (`AI=2G/b`); tensor cores attack the *schedule*, not the bytes

The model can't see this step: same bytes → same `AI=2G/b`, same HBM floor. The prediction is a schedule claim — the GEMM should lower µs/tok and RAISE %HBM vs Cut 1 at a given G (closing the gap), though at G=8 the WMMA tile is half-empty (8 real rows / 16) so 2a captures the direction, not the peak.

In [3]:
from roofline.archs import get_arch
from roofline.model import estimate
arch = get_arch('sm_80')
print(f"{'G':>3} | {'AI=2G/b':>8} | {'limiter':>7} | {'t_hbm floor':>12}  (identical to Cut 1 — bytes unchanged)")
for G in (1,2,4,8,16,32):
    e = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp16', G=G)
    print(f'{G:>3} | {e.arithmetic_intensity:8.1f} | {e.limiter.upper():>7} | {e.t_hbm*1e3:9.4f}ms')
print('\nPrediction: tensor cores lower us/tok + raise %HBM vs Cut 1 at G>=4 (GEMM closes the per-CTA gap).')


  G |  AI=2G/b | limiter |  t_hbm floor  (identical to Cut 1 — bytes unchanged)
  1 |      1.0 |     HBM |    0.1317ms
  2 |      2.0 |     HBM |    0.0658ms
  4 |      4.0 |     HBM |    0.0329ms
  8 |      8.0 |     HBM |    0.0165ms
 16 |     16.0 |     HBM |    0.0082ms
 32 |     31.9 |     HBM |    0.0041ms

Prediction: tensor cores lower us/tok + raise %HBM vs Cut 1 at G>=4 (GEMM closes the per-CTA gap).


## 3. Build v8_gqa_tc (JIT, Turing WMMA)

In [4]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v8_gqa_tc')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
mod = build_kernel('v8_gqa_tc')
print('built:', mod)


built: <module 'fa_v8_gqa_tc' from '/root/.cache/torch_extensions/py312_cu128/fa_v8_gqa_tc/fa_v8_gqa_tc.so'>


## 4. Correctness gate — v8_gqa_tc (Gate 1 of 2)

Same GQA cases as Cut 1 (decode `G∈{1,2,4,8}` × non-multiple `N_k` × causal+offset, idle/pad `G=3` + multi-tile `G=16`, square reduction), now on the WMMA backend. Oracle = `sdpa_reference_gqa`, tol 2e-2.

In [5]:
!python -m pytest tests/test_correctness.py -k "v8_gqa_tc" -q


......................................                                   [100%]
38 passed, 164 deselected in 2.73s


## 5. The A/B — v8_gqa_tc (tensor cores) vs v8_gqa (Cut 1, CUDA cores), same G-sweep

Identical workload, both backends. Compare `us/tok` and `%HBM` per G — does the GEMM beat the warp-shuffle GEMV, and by how much as G crosses the M<16→M≥16 (pad→full) line?

In [6]:
print('=== Cut 2a: v8_gqa_tc (Turing WMMA tensor cores) ===')
!python -m bench.harness --backend v8_gqa_tc --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== Cut 1: v8_gqa (CUDA cores) — same workload, for the A/B ===')
!python -m bench.harness --backend v8_gqa --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32


=== Cut 2a: v8_gqa_tc (Turing WMMA tensor cores) ===
# device: Tesla T4 (sm_75)  clock~360/1590MHz  backend=v8_gqa_tc  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
[1/3] c++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fa_v7_paged -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.12 -fPIC -std=c++17 -c /content/flashattention-cuda/kernels/v7_paged/binding.cpp -o binding.o 
[2/3] /usr/local/cuda/bin/nvcc -MD -MF paged_attention.cuda.o.d -DTORCH_EXTENSION_NAME=fa_v7_paged -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -is

## 6. Reclaim-SDPA-at-batch on tensor cores (G=8)

Does the tensor-core path still beat SDPA across the serving batch range (it must at least match Cut 1)?

In [7]:
!python -m bench.harness --backend v8_gqa_tc --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64


# device: Tesla T4 (sm_75)  clock~585/1590MHz  backend=v8_gqa_tc  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
ninja: no work to do.
     1x8x1x64/8192 G8 |   0.563/  0.628 |    70.43 |   1.2% |    4.76x |    1.08x | HBM (~0.01ms)
    1x8x1x128/8192 G8 |   0.695/  0.733 |    86.85 |   1.9% |    2.66x |    1.36x | HBM (~0.01ms)
     8x8x1x64/8192 G8 |   1.446/  1.552 |    22.59 |   3.6% |    3.66x |    3.36x | HBM (~0.05ms)
    8x8x1x128/8192 G8 |   2.657/  2.805 |    41.51 |   3.9% |    3.00x |    3.03x | HBM (~0.10ms)
    16x8x1x64/8192 G8 |   2.677/  2.840 |    20.91 |   3.9% |    3.98x |    3.64x | HBM (~0.10ms)
   16x8x1x128/8192 G8 |   5.028/  5.212 |    39.28 |   4.2% |    3.15x |    3.20x | HBM (~0.21ms)
    32x8x1x64/8192 G8 |   7.611/  7.731 |    29.73 |   2.8% |    2.67x |    2.53x | HBM (~0.21ms)
   32x8x1x128/8192 G8 |  15.020/ 15.206 |    58.67 |   2.8% |    2